# GemCol Evaluation: Phase 6 (BEIR Zero-Shot Benchmarks)
This notebook runs the full TriRank pipeline (BM25 $\rightarrow$ BGE-large $\rightarrow$ RRF $\rightarrow$ ColBERTv2) on the BEIR zero-shot datasets: `nfcorpus`, `scifact`, `trec-covid`, and `arguana`.

**Instruction:** Change the `DATASET` variable below and "Run All" to process each dataset. Results are saved to `experiments.json` automatically.

In [ ]:
!pip install -q beir

In [ ]:
import os
import sys
import json
import torch
import torch.nn as nn
from tqdm import tqdm
import bm25s
from sentence_transformers import SentenceTransformer
from transformers import BertPreTrainedModel, BertModel, AutoTokenizer, BertConfig
from huggingface_hub import hf_hub_download
import safetensors.torch

# Import BEIR
from beir import util
from beir.datasets.data_loader import GenericDataLoader

sys.path.insert(0, '/workspace/gemcol_evaluation')
from utils import rrf_fusion, evaluate_run, print_metrics, LatencyProfiler, ExperimentTracker

# ---------------------------------------------------------
# CHANGE THIS TO RUN DIFFERENT DATASETS
# Available options: 'nfcorpus', 'scifact', 'trec-covid', 'arguana'
DATASET = 'nfcorpus'
# ---------------------------------------------------------

## 1. Load Dataset

In [ ]:
print(f"Downloading and loading BEIR dataset: {DATASET}...")
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET}.zip"
out_dir = os.path.join("/workspace/data", DATASET)
data_path = util.download_and_unzip(url, out_dir)

# Load corpus, queries, and qrels
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

# Extract arrays for processing
doc_ids = list(corpus.keys())
corpus_texts = [corpus[doc_id].get("title", "") + " " + corpus[doc_id].get("text", "") for doc_id in doc_ids]

query_ids = list(queries.keys())
query_texts = [queries[qid] for qid in query_ids]

print(f"Loaded {len(corpus_texts)} documents and {len(query_texts)} queries.")

## 2. Stage 1: BM25 Sparse Retrieval

In [ ]:
print("Indexing corpus with BM25...")
retriever = bm25s.BM25()
corpus_tokens = bm25s.tokenize(corpus_texts, stopwords="en")
retriever.index(corpus_tokens)

print("Retrieving BM25 candidates...")
query_tokens = bm25s.tokenize(query_texts, stopwords="en")
results, scores = retriever.retrieve(query_tokens, k=100)

bm25_run = {}
for i, qid in enumerate(query_ids):
    bm25_run[qid] = [doc_ids[idx] for idx in results[i]]

metrics = evaluate_run(bm25_run, qrels)
print_metrics(metrics, system_name=f"BM25 Baseline ({DATASET})")

del corpus_tokens, query_tokens, retriever
import gc
gc.collect()

## 3. Stage 2: BGE-large Dense Retrieval

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Loading BGE-large-en-v1.5...")
bge_model = SentenceTransformer('BAAI/bge-large-en-v1.5', device=device)

print("Encoding queries and corpus...")
# Note: BEIR datasets are much smaller than MS MARCO, so we can encode the corpus dynamically without chunks.
query_embeddings = bge_model.encode(query_texts, convert_to_tensor=True, normalize_embeddings=True, batch_size=256)
corpus_embeddings = bge_model.encode(corpus_texts, convert_to_tensor=True, normalize_embeddings=True, batch_size=512, show_progress_bar=True)

print("Computing Dense Exact Search...")
similarities = torch.matmul(query_embeddings, corpus_embeddings.T)
top_k_scores, top_k_indices = torch.topk(similarities, k=100, dim=1)

top_k_indices_cpu = top_k_indices.cpu().numpy()
bge_run = {}
for i, qid in enumerate(query_ids):
    bge_run[qid] = [doc_ids[idx] for idx in top_k_indices_cpu[i]]

metrics = evaluate_run(bge_run, qrels)
print_metrics(metrics, system_name=f"BGE Exact ({DATASET})")

# Clear memory
del bge_model, query_embeddings, corpus_embeddings, similarities, top_k_scores, top_k_indices
torch.cuda.empty_cache()

## 4. Stage 3: RRF Fusion

In [ ]:
print("Applying Reciprocal Rank Fusion...")
fused_run = rrf_fusion([bm25_run, bge_run], k=60, top_k=100)
metrics = evaluate_run(fused_run, qrels)
print_metrics(metrics, system_name=f"RRF Fusion ({DATASET})")

## 5. Stage 4: ColBERTv2 Reranking

In [ ]:
class ColBERT(nn.Module):
    def __init__(self):
        super().__init__()
        config = BertConfig.from_pretrained("colbert-ir/colbertv2.0")
        self.bert = BertModel(config)
        self.linear = nn.Linear(config.hidden_size, 128, bias=False)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        embeddings = self.linear(outputs.last_hidden_state)
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=-1)
        return embeddings

print("Loading ColBERTv2 natively...")
tokenizer = AutoTokenizer.from_pretrained("colbert-ir/colbertv2.0")
colbert_model = ColBERT()
model_path = hf_hub_download("colbert-ir/colbertv2.0", "model.safetensors")
colbert_model.load_state_dict(safetensors.torch.load_file(model_path), strict=False)
colbert_model = colbert_model.to(device)
colbert_model.eval()

def encode_query(query_text):
    q_text = "[unused0] " + query_text
    inputs = tokenizer(q_text, return_tensors="pt", max_length=32, truncation=True, padding="max_length").to(device)
    inputs['input_ids'][inputs['input_ids'] == tokenizer.pad_token_id] = tokenizer.mask_token_id
    with torch.no_grad():
        return colbert_model(**inputs)

def encode_documents_and_score(Q, doc_texts):
    d_texts = ["[unused1] " + d for d in doc_texts]
    inputs = tokenizer(d_texts, return_tensors="pt", max_length=150, truncation=True, padding=True).to(device)
    with torch.no_grad():
        D = colbert_model(**inputs)
    sim = torch.einsum("iqd,bjd->bqj", Q, D)
    d_mask = inputs['attention_mask'].unsqueeze(1)
    sim = sim.masked_fill(d_mask == 0, -10000.0)
    max_scores, _ = torch.max(sim, dim=2)
    return torch.sum(max_scores, dim=1)

colbert_run = {}
for qid in tqdm(query_ids, desc=f"ColBERT Reranking ({DATASET})"):
    query_text = queries[qid]
    candidate_doc_ids = fused_run[qid]
    candidate_texts = [corpus[doc_id].get("title", "") + " " + corpus[doc_id].get("text", "") for doc_id in candidate_doc_ids]
    
    Q = encode_query(query_text)
    scores = encode_documents_and_score(Q, candidate_texts)
    
    sorted_pairs = sorted(zip(candidate_doc_ids, scores.cpu().numpy()), key=lambda x: x[1], reverse=True)
    colbert_run[qid] = [doc_id for doc_id, score in sorted_pairs]

metrics = evaluate_run(colbert_run, qrels)
print_metrics(metrics, system_name=f"TriRank Final ({DATASET})")

# Log the final TriRank score to experiments.json
tracker = ExperimentTracker("/workspace/results/experiments.json")
tracker.log(f"TriRank_BEIR_{DATASET}", config={"dataset": DATASET}, metrics={
    "ndcg@10": metrics["ndcg@10"],
    "mrr@10": metrics["mrr@10"]
})
print(f"\n✅ Fully evaluated {DATASET} and logged to experiments.json!")
print("--> You can now change DATASET at the top and click 'Run All' to process the next one.")